<a href="https://colab.research.google.com/github/zsk39/DECISION-563Q-Session-7-Demo/blob/main/Session7_demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Session 7: Integrated Python case study

# Suspicious Transaction Detector

Imagine you’ve been hired as a data analyst at a credit card company investigating fraudulent activity. Your job is to get their basic suspicious transaction detector up and running.

The system should:
- Analyze historical transaction data
- Detect unusual or suspicious behavior
- Flag and output these transactions to a file called `flagged_transactions.csv`

However, the provided code is **buggy**! Your job is to fix the major bugs using your programming skills and (if needed) your prefered flavor of GenAI as an assistant.

**Primary issue to fix:**
- The `is_geo_jump` method currently uses a `city_coords` variable, but this is not defined.
- You must update the function to pull **geolocation data from the provided database** in order to compute distances.

There may be additional bugs to address.

Once fixed, the code should:
- Flag suspicious transactions
- Save them to `flagged_transactions.csv`
- Print a message confirming how many transactions were flagged

In [34]:
# imported numpy which was initially missing
import numpy as np
import sqlite3
import csv
from datetime import datetime
from collections import defaultdict
from geopy.distance import geodesic

In [38]:
KM_DIST = {}

class Transaction:
    def __init__(self, row, user_history, db_conn):
        self.transaction_id = row["transaction_id"]
        self.user_id = row["user_id"]
        self.amount = row["amount"]
        self.timestamp = datetime.fromisoformat(row["timestamp"])
        self.merchant = row["merchant"]
        self.location = row["location"]
        self.user_history = user_history
        self.db_conn = db_conn

    def is_high_value(self):
        amounts = self.user_history.get("amounts", [])
        if len(amounts) < 5:
            return False
        threshold = np.percentile(amounts, 99)
        return self.amount > threshold # corrected misspelling of threshold (initially "theshold")

    def is_geo_jump(self):
        locs = self.user_history.get("locations", [])
        times = self.user_history.get("timestamps", [])
        if not locs or not times:
            return False
        prev_location = locs[-1]
        prev_time = times[-1]

        if prev_location == self.location:
            return False

        city_pair = tuple(sorted([prev_location, self.location]))
        distance_km = 0

        if city_pair in KM_DIST:
            distance_km = KM_DIST[city_pair]
        else:
            def get_coords(city):
                city, state = city.split(',')
                #print(city, state)
                cur = self.db_conn.cursor()
                cur.execute("SELECT lat, lng FROM cities WHERE city = ? AND state_id = ?", (city, state.strip()))
                result = cur.fetchone()
                #print(result)
                return result if result else (None, None)

            coords1 = get_coords(prev_location)
            coords2 = get_coords(self.location)
            if not coords1 or not coords2:
                return False

            distance_km = geodesic(coords1, coords2).kilometers
            KM_DIST[city_pair] = distance_km
        time_diff_hr = (self.timestamp - prev_time).total_seconds() / 3600

        return distance_km > 400 and time_diff_hr < distance_km / 500  # Faster than a jet

    def is_unusual_vendor(self):
        merchants = self.user_history.get("merchants", set())
        return len(merchants) >= 5 and self.merchant not in merchants

    def is_suspicious(self): #added missing colon
        reasons = []
        if self.is_high_value():
            reasons.append("High value")
        if self.is_geo_jump():
            reasons.append("Unrealistic geo jump")
        if self.is_unusual_vendor():
            reasons.append("New vendor")
        return reasons


In [39]:
# misspelled fraud as "fraub" for the default database name
def run_pipeline(db_path='fraud.db', cutoff="2024-07-15 00:00:00", output_file='flagged_transactions.csv'):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute("SELECT * FROM transactions ORDER BY timestamp")
    rows = cursor.fetchall()
    col_names = [desc[0] for desc in cursor.description]

    cutoff_dt = datetime.fromisoformat(cutoff)
    user_history = defaultdict(lambda: {"amounts": [], "locations": [], "timestamps": [], "merchants": set()})
    flagged = []

    for row in rows:
        record = dict(zip(col_names, row))
        txn_time = datetime.fromisoformat(record["timestamp"])
        user_id = record["user_id"]
        profile = user_history[user_id]

        txn = Transaction(record, profile, conn)

        if txn_time < cutoff_dt:
            profile["amounts"].append(txn.amount)
            profile["locations"].append(txn.location)
            profile["timestamps"].append(txn.timestamp)
            profile["merchants"].add(txn.merchant)
        else:
            reasons = txn.is_suspicious()
            if reasons:
                flagged.append([txn.transaction_id, txn.user_id, txn.timestamp.isoformat(), "; ".join(reasons)])

    conn.close()

    with open(output_file, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["transaction_id", "user_id", "timestamp", "reasons"])
        for row in flagged:
            writer.writerow(row)

    print(f"Flagged {len(flagged)} transactions. Output saved to {output_file}.")

In [40]:
run_pipeline()

Flagged 6867 transactions. Output saved to flagged_transactions.csv.
